In [0]:
# CLEANUP JOB


from datetime import datetime, timedelta

# Configuration
catalog = 'wikimedia_db'
uc_schema_raw_events = 'raw_events'
retention_days = 2  # Keep data for 2 days (adjust as needed)

print(f" CLEANUP JOB STARTED")
print(f"   Catalog: {catalog}")
print(f"   Schema: {uc_schema_raw_events}")
print(f"   Retention: {retention_days} days")
print("=" * 60)

In [0]:

def cleanup_old_volumes(catalog: str, schema: str, retention_days: int):
    
    cutoff_date = datetime.now() - timedelta(days=retention_days)
    volumes_path = f"/Volumes/{catalog}/{schema}/"
    
    deleted_count = 0
    kept_count = 0
    error_count = 0
    
    print(f" Cutoff date: {cutoff_date.strftime('%Y-%m-%d')}")
    print(f"   (Deleting volumes older than this date)\n")
    
    try:
        # List all items in the schema
        all_items = dbutils.fs.ls(volumes_path)
        
        for item in all_items:
            volume_name = item.name.rstrip('/')
            
            # Check if it's a temporary event volume
            if volume_name.startswith('events_tmp_'):
                try:
                    # Extract date from volume name (events_tmp_YY_MM_DD)
                    date_parts = volume_name.split('_')[-3:]
                    volume_date = datetime.strptime('_'.join(date_parts), '%y_%m_%d')
                    
                    if volume_date < cutoff_date:
                        print(f"  DELETING: {volume_name}")
                        print(f"    Date: {volume_date.strftime('%Y-%m-%d')} (expired)")
                        
                        # Drop the volume
                        spark.sql(f"DROP VOLUME IF EXISTS {catalog}.{schema}.{volume_name}")
                        deleted_count += 1
                        
                    else:
                        print(f" KEEPING: {volume_name}")
                        print(f"    Date: {volume_date.strftime('%Y-%m-%d')} (recent)")
                        kept_count += 1
                        
                except ValueError as e:
                    print(f"  SKIPPED: {volume_name}")
                    print(f"    Reason: Invalid date format - {str(e)}")
                    error_count += 1
                    
                except Exception as e:
                    print(f" ERROR: {volume_name}")
                    print(f"    Reason: {str(e)}")
                    error_count += 1
                    
            else:
                print(f" SKIPPED: {volume_name} (not a temp volume)")
    
    except Exception as e:
        print(f" FATAL ERROR while listing volumes: {str(e)}")
        return {
            'success': False,
            'error': str(e),
            'deleted': 0,
            'kept': 0,
            'errors': 0
        }
    
    print("\n" + "=" * 60)
    print(" CLEANUP SUMMARY:")
    print(f"    Volumes deleted: {deleted_count}")
    print(f"    Volumes kept: {kept_count}")
    print(f"     Errors: {error_count}")
    print(f"    Retention period: {retention_days} days")
    print("=" * 60)
    
    return {
        'success': True,
        'deleted': deleted_count,
        'kept': kept_count,
        'errors': error_count,
        'retention_days': retention_days,
        'cutoff_date': cutoff_date.strftime('%Y-%m-%d')
    }


# Execute cleanup
print("  WARNING: This will permanently delete old data volumes!")
print(f"   Retention period: {retention_days} days\n")

# Uncomment the line below to actually run the cleanup
result = cleanup_old_volumes(catalog, uc_schema_raw_events, retention_days)
# Display results
if result['success']:
    print("\n CLEANUP COMPLETED SUCCESSFULLY")
    
    # Create summary for logging/monitoring
    summary_df = spark.createDataFrame([{
        'execution_time': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'volumes_deleted': result['deleted'],
        'volumes_kept': result['kept'],
        'errors': result['errors'],
        'retention_days': result['retention_days'],
        'cutoff_date': result['cutoff_date']
    }])
    
    display(summary_df)
else:
    print("\n CLEANUP FAILED")
    print(f"   Error: {result.get('error', 'Unknown error')}")
